# Axral Vocal AI — Kaggle One-cell pipeline
**入力は `HF_DATA_URL` だけ**。Hugging Face Dataset取得→文字コードに配慮した音源監査→pytest→承認済みラベルmanifestがある場合の学習・評価→成果物ZIPまで1セルで実行します。
> 学習対象は現時点の実験用メタデータ・ピッチ差分モデルです。UTAU WAVだけから歌唱表現の正解ラベルは生成できないため、ラベル付きJSONLが無ければ学習を安全に停止します。音声生成モデルの学習ではありません。

In [ ]:
# ============================================================
# AXRAL VOCAL AI — ONE CELL KAGGLE PIPELINE
# 設定は HF_DATA_URL だけ。取得→文字コード対策→監査→pytest→条件付き学習・評価→ZIP
# ============================================================
import os, re, sys, json, subprocess, traceback, zipfile
from pathlib import Path

# ★ここだけ変更：Hugging Face の Dataset URL（例 https://huggingface.co/datasets/OWNER/REPO）
HF_DATA_URL = "https://huggingface.co/datasets/SEKOIA2900/VO-US"
PROJECT = Path("/kaggle/working/Axral_vocal_AI")
DATA_ROOT = Path("/kaggle/working/hf_voicebanks")
OUT = Path("/kaggle/working/axral_run")
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "pipeline.log"

def log(msg):
    print(msg)
    with LOG.open("a", encoding="utf-8") as f: f.write(str(msg) + "\n")

def run(cmd, check=True):
    log("$ " + " ".join(map(str, cmd)))
    p = subprocess.run(cmd, cwd=PROJECT, text=True, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, errors="replace")
    with LOG.open("a", encoding="utf-8") as f: f.write(p.stdout)
    print(p.stdout[-5000:])
    if check and p.returncode: raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

LOG.write_text("", encoding="utf-8")
try:
    assert HF_DATA_URL.startswith("https://huggingface.co/"), "HF_DATA_URL must be a Hugging Face URL"
    if not PROJECT.exists():
        log("Project checkout missing; cloning Axral_vocal_AI main branch.")
        clone = subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/nebula-INE/Axral_vocal_AI.git", str(PROJECT)],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, errors="replace"
        )
        log(clone.stdout)
        if clone.returncode:
            raise RuntimeError(f"Git clone failed ({clone.returncode})")
    run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[training]"])
    run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])

    # Dataset URL -> repository ID -> download/cache (private datasets use Kaggle Secrets HF_TOKEN)
    m = re.search(r"huggingface\.co/datasets/([^/?#]+/[^/?#]+)", HF_DATA_URL)
    if not m: raise ValueError("Dataset URL must look like huggingface.co/datasets/owner/name")
    dataset_id = m.group(1)
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    from huggingface_hub import snapshot_download
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None  # Public datasets do not need a token.
    log(f"[1/5] Download dataset: {dataset_id}")
    snapshot_download(repo_id=dataset_id, repo_type="dataset", local_dir=str(DATA_ROOT),
                     token=token)

    # Encoding safety: audit text is decoded by project reader (BOM/UTF-16/UTF-8/CP932).
    # Keep original voicebank files untouched; no destructive re-encoding.
    log("[2/5] Locate voicebank roots")
    banks = sorted({p.parent for p in DATA_ROOT.rglob("oto.ini")})
    log(f"oto.ini roots: {len(banks)}")
    if not banks: raise RuntimeError("No oto.ini found. Check HF dataset layout/download.")

    log("[3/5] Voicebank audit (JSON per bank)")
    audit_dir = OUT / "audits"; audit_dir.mkdir(exist_ok=True)
    for i, bank in enumerate(banks, 1):
        safe = re.sub(r"[^A-Za-z0-9._-]+", "_", bank.relative_to(DATA_ROOT).as_posix())[:100] or f"bank_{i}"
        target = audit_dir / f"{i:03d}_{safe}.json"
        run([sys.executable, "-m", "axral_vocal_ai.voicebank_audit_cli",
             str(bank), "--output", str(target)], check=False)
        log(f"audit {i}/{len(banks)}: {bank} -> {target}")

    log("[4/5] Tests")
    test_rc = run([sys.executable, "-m", "pytest", "-q"], check=False)
    log(f"pytest exit={test_rc}")

    log("[5/5] Discover approved labelled training manifests")
    manifests = sorted({p for p in DATA_ROOT.rglob("*.jsonl") if p.is_file()})
    log(f"JSONL candidates: {len(manifests)}")
    eligible = []
    validation_failures = []
    training_status = "blocked_no_approved_labelled_manifest"
    model_report = OUT / "models" / "pitch_delta_report.json"

    if test_rc != 0:
        training_status = "blocked_tests_failed"
        log("Training blocked because pytest failed.")
    else:
        # Validate every candidate using the repository's strict permission,
        # label, split-isolation, path-redaction, and numeric-range checks.
        for manifest in manifests:
            rc = run([
                sys.executable, "training/validate_dataset.py",
                str(manifest), "--require-labeled"
            ], check=False)
            if rc == 0:
                eligible.append(manifest)
            else:
                validation_failures.append({
                    "manifest": manifest.relative_to(DATA_ROOT).as_posix(),
                    "exit_code": rc
                })

        if len(eligible) == 1:
            log(f"Using validated manifest: {eligible[0].relative_to(DATA_ROOT)}")
            model_report.parent.mkdir(parents=True, exist_ok=True)
            # This is the existing metadata-only ridge MVP, not a waveform trainer.
            # Fixed alpha=1.0; evaluate the isolated test voicebank once at the end.
            run([
                sys.executable, "training/train_pitch_model.py",
                str(eligible[0]), "--output", str(model_report), "--evaluate-test"
            ])
            training_status = "metadata_pitch_model_trained_and_tested"
        elif len(eligible) > 1:
            training_status = "blocked_multiple_eligible_manifests"
            log("Training blocked: multiple eligible manifests found; refusing to choose one implicitly.")
        else:
            log("No eligible labelled JSONL manifest found.")
            log("UTAU oto.ini/WAV inventory does not contain aligned score-to-pitch-delta labels.")
            log("No model training was attempted; provide a reviewed pitch_training_manifest.jsonl in the HF dataset.")

    summary = {
        "dataset": dataset_id,
        "voicebank_roots": len(banks),
        "audit_files": len(list(audit_dir.glob("*.json"))),
        "pytest_exit": test_rc,
        "jsonl_candidates": len(manifests),
        "eligible_manifests": len(eligible),
        "manifest_validation_failures": validation_failures,
        "training": training_status,
        "model_report": str(model_report) if model_report.exists() else None,
        "training_scope": "metadata_only_pitch_delta_ridge_mvp",
        "waveform_model_training": "not_implemented",
        "gpu_training_used": False
    }
    (OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    log(json.dumps(summary, ensure_ascii=False))
except Exception:
    log(traceback.format_exc())
    raise
finally:
    log(f"LOG: {LOG}")
    log(f"OUTPUT: {OUT}")
    archive = OUT.parent / "axral_run_artifacts.zip"
    log(f"ARTIFACT_ZIP: {archive}")
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for artifact in OUT.rglob("*"):
            if artifact.is_file():
                bundle.write(artifact, artifact.relative_to(OUT.parent))
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(archive)))
    except Exception as display_error:
        log(f"Artifact link display unavailable: {display_error}")

